# 04 Root Finding

[![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE)
[![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)

## The Lens: Equilibrium is a Zero

**What problem are we solving?**  
In economics, equilibrium is defined by balance. 
*   **Market Clearing:** Supply equals Demand $\implies S(p) - D(p) = 0$.
*   **Arbitrage:** Returns are equalized $\implies R_A - R_B = 0$.
*   **Steady State:** Capital doesn't change $\implies s f(k) - \delta k = 0$.

These are all **root-finding problems**. We are looking for the variable $x$ such that $f(x) = 0$.

**Why this method?**  
Analytical solutions ($x = \frac{-b \pm \dots}{2a}$) are rare. We need numerical algorithms that can hunt down the zero for any function.
*   **Bisection:** Slow but guaranteed. Good for 1D problems where you know the bounds.
*   **Newton-Raphson:** Fast but risky. Requires derivatives.
*   **Brent's Method:** The industrial standard (hybrid). Safest for 1D.
*   **Homotopy Continuation:** A robust method for difficult systems of equations.

## Learning Objectives

By the end of this notebook, you will be able to:
1.  **Formulate** economic equilibrium problems as root-finding tasks.
2.  **Apply** the Fixed Point Iteration method and understand the Contraction Mapping Theorem.
3.  **Implement** Bisection and Newton's method.
4.  **Solve** for market clearing prices in a General Equilibrium model.
5.  **Utilize** Homotopy Continuation to solve systems where Newton's method fails.

## Prerequisites

*   **02-Numerical-Methods/03_Numerical_Differentiation.ipynb**: Newton's method relies on derivatives.

In [ ]:
# === Environment Setup ===
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import brentq, newton, root

# --- Configuration ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'figure.dpi': 120})
np.set_printoptions(suppress=True, precision=4, linewidth=120)
warnings.filterwarnings('ignore')

## 1. Fixed-Point Theory

Many economic problems are naturally phrased as finding a **fixed point**: $x = g(x)$.
*   Example: In the Solow model, $k_{t+1} = s k_t^\alpha + (1-\delta)k_t$. The steady state is $k^* = g(k^*)$.

**Fixed Point Iteration:** Guess $x_0$, then compute $x_{t+1} = g(x_t)$.

**Contraction Mapping Theorem (Banach):** If $g$ is a "contraction" (Lipschitz constant $\beta < 1$, i.e., slope strictly between -1 and 1 everywhere), this iteration is guaranteed to converge to a unique fixed point. This provides a *constructive* method to find the fixed point.

**Brouwer's Fixed Point Theorem:** A continuous function mapping a compact, convex set to itself has at least one fixed point. This guarantees *existence*, but not uniqueness, and doesn't tell us how to find it (non-constructive). It is crucial for proving existence of Nash Equilibria.

### Visualizing Convergence: The Cobweb Plot
For $x_{t+1} = g(x_t)$, we can visualize the path by drawing a line from $(x_t, x_t)$ vertically to the curve $(x_t, g(x_t))$, and then horizontally to $(g(x_t), g(x_t)) = (x_{t+1}, x_{t+1})$. This creates a "cobweb" or staircase pattern that spirals into the fixed point (if stable) or spirals out (if unstable).

## 2. Root-Finding Algorithms

Any fixed point problem $x = g(x)$ can be rewritten as a root finding problem $f(x) = x - g(x) = 0$.

### 2.1 Bisection (Bracketing)
If $f(a) < 0$ and $f(b) > 0$, there must be a root between $a$ and $b$ (Intermediate Value Theorem). 
1.  Check midpoint $m = (a+b)/2$.
2.  If $f(m) > 0$, the root is in $[a, m]$. Else in $[m, b]$.
3.  Repeat.

**Pros:** Guaranteed convergence. **Cons:** Slow (Linear convergence).

### 2.2 Newton-Raphson (Open)
Use the tangent line to jump to the root.
$$ x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)} $$

**Derivation:** Newton's method comes from the first-order Taylor expansion: $f(x) \approx f(x_n) + f'(x_n)(x - x_n)$. Setting this linear approximation to zero gives the update rule.

### Halley's Method (Cubic Convergence)
If we keep the second-order term in the Taylor series, we get **Halley's Method**. It converges much faster (cubic rate) but requires the second derivative $f''(x)$.
$$ x_{n+1} = x_n - \frac{2 f(x_n) f'(x_n)}{2 [f'(x_n)]^2 - f(x_n) f''(x_n)} $$

### Broyden's Method (Quasi-Newton)
For systems of equations, calculating the Jacobian $J$ and solving the linear system at every step is expensive ($O(N^3)$). Broyden's method approximates the Jacobian and updates it using a rank-1 formula (the "Secant equation" for matrices).
$$ J_{n+1} = J_n + \frac{(y_n - J_n s_n)s_n^T}{s_n^T s_n} $$
where $s_n = x_{n+1} - x_n$ and $y_n = F(x_{n+1}) - F(x_n)$. This allows for $O(N^2)$ updates. It converges superlinearly (slower than quadratic, but much cheaper per step).

**Acceleration Methods: Anderson Mixing**
For fixed-point problems $x = g(x)$, simple iteration can be slow. **Anderson Acceleration** uses a weighted average of the last $m$ iterates to minimize the residual $x - g(x)$. It acts like a Quasi-Newton method but doesn't require computing a Jacobian. It is highly effective in solving Dynamic Programming problems (Value Function Iteration).

**Proof of Quadratic Convergence:**
Let $e_n = x_n - x^*$ be the error. Taylor expand $f(x^*)$ around $x_n$:
$0 = f(x^*) = f(x_n) + f'(x_n)(x^* - x_n) + \frac{f''(\xi)}{2}(x^* - x_n)^2$
Divide by $f'(x_n)$:
$0 = \frac{f(x_n)}{f'(x_n)} - e_n + \frac{f''(\xi)}{2f'(x_n)}e_n^2$
Since Newton's update is $x_{n+1} = x_n - f(x_n)/f'(x_n)$, we have $e_{n+1} = x_{n+1} - x^* = e_n - f(x_n)/f'(x_n)$.
Substituting:
$e_{n+1} \approx \frac{f''(x^*)}{2f'(x^*)} e_n^2$. The error at the next step is proportional to the *square* of the current error.

**Halley's Method (3rd Order):**
If we keep the 2nd order term in the Taylor series, we get Halley's method (cubic convergence):
$$ x_{n+1} = x_n - \frac{2f(x_n)f'(x_n)}{2[f'(x_n)]^2 - f(x_n)f''(x_n)} $$

**Pros:** Extremely fast (Quadratic convergence). **Cons:** Can diverge if initial guess is bad or derivative is zero.

**Basins of Attraction:** The set of initial guesses that converge to a specific root is called its basin of attraction. For complex functions, these basins can be fractal (e.g., the Newton fractal).

### 2.3 Brent's Method (The Best of Both)
`scipy.optimize.brentq` combines bisection (safety) with secant method (speed). **Always use this for scalar problems.**

### Application: Yield to Maturity

The price of a bond is the present value of its payments. The Yield to Maturity (YTM) is the interest rate $y$ that makes the present value equal to the market price $P_{market}$.

$$ P(y) - P_{market} = 0 $$

In [ ]:
def bond_excess_price(y, price, coupon, face, T):
    # PV of coupons
    pv_coupons = sum(coupon / (1+y)**t for t in range(1, T+1))
    # PV of face value
    pv_face = face / (1+y)**T
    return (pv_coupons + pv_face) - price

# Bond Params
market_price = 950
coupon = 50
face = 1000
T = 10

# Find YTM using Brent's method
# We know yield must be between 0% and 20%
ytm = brentq(bond_excess_price, 0.0, 0.2, args=(market_price, coupon, face, T))

print(f"Market Price: ${market_price}")
print(f"Yield to Maturity: {ytm:.2%}")

## 3. Systems of Equations: Market Equilibrium

For $N$ markets, we have a system $F(\mathbf{p}) = \mathbf{0}$, where $\mathbf{p}$ is a vector of prices.

**Example: General Equilibrium with CES Utility**
Two agents (A, B), two goods (1, 2). 
We want to find the relative price $p_1$ (normalize $p_2=1$) such that Excess Demand for Good 1 is zero.

In [ ]:
# Agent A: Prefers Good 1
alpha_A, rho_A = 0.7, -1.0 # CES params
endow_A = np.array([4.0, 1.0])

# Agent B: Prefers Good 2
alpha_B, rho_B = 0.3, -1.0
endow_B = np.array([1.0, 4.0])

def get_demand(p1, income, alpha, rho):
    # Marshallian demand for Good 1 from CES FOCs
    # Derived from: Max (alpha*c1^rho + (1-alpha)c2^rho)^(1/rho)
    p2 = 1.0
    sigma = 1 / (1 - rho)
    term = (alpha / (1-alpha) * p2 / p1)**sigma
    c1 = income / (p1 + p2/term)
    return c1

def excess_demand(p1):
    p2 = 1.0
    # 1. Calculate Incomes
    inc_A = p1*endow_A[0] + p2*endow_A[1]
    inc_B = p1*endow_B[0] + p2*endow_B[1]
    
    # 2. Calculate Demands for Good 1
    c1_A = get_demand(p1, inc_A, alpha_A, rho_A)
    c1_B = get_demand(p1, inc_B, alpha_B, rho_B)
    
    # 3. Supply
    supply_1 = endow_A[0] + endow_B[0]
    
    return (c1_A + c1_B) - supply_1

# Solve for equilibrium price
p1_star = brentq(excess_demand, 0.1, 10.0)

print(f"Equilibrium Price p1: {p1_star:.4f}")
print(f"Excess Demand at p1*: {excess_demand(p1_star):.2e}")

## 4. Homotopy Continuation

Finding roots for systems $F(x)=0$ is hard. If your initial guess is far off, Newton's method explodes.

**The Idea:** Start with an easy problem $G(x)=0$ (where you know the solution) and slowly transform it into the hard problem $F(x)=0$.

$$ H(x, t) = (1-t)G(x) + tF(x) $$

1.  Start at $t=0$ with known solution $x_0$.
2.  Increase $t$ slightly (e.g., $t=0.1$). Use $x_0$ as guess to solve $H(x, 0.1)=0$.
3.  Repeat until $t=1$.

This "path-following" keeps you in the basin of attraction of the root.

In [ ]:
# Difficult problem: x^3 - 1 = 0 (finding real root x=1 is easy, but let's pretend it's hard)
# Let's assume we don't know x=1. 
# Easy problem: x - 0.5 = 0 (Solution x=0.5)

def F(x): return x**3 - 1
def G(x): return x - 0.5

def H(x, t):
    return (1-t)*G(x) + t*F(x)

x_curr = 0.5 # Known solution for t=0
t_steps = np.linspace(0, 1, 11)

print("Homotopy Path:")
for t in t_steps:
    # Solve H(x, t) = 0 using previous x as guess
    # Use newton for the step
    x_curr = newton(lambda x: H(x, t), x0=x_curr)
    print(f"  t={t:.1f}, Root={x_curr:.4f}")

print(f"Final Solution for F(x)=0: {x_curr:.4f}")

## Summary

**Key Takeaways:**
*   **Everything is a Root:** Equilibrium, steady states, and arbitrage conditions are all $f(x)=0$.
*   **Scalar? Use Brent:** `brentq` is the default choice for 1D problems.
*   **Vector? Use Newton:** For systems, use `scipy.optimize.root` (which uses Newton-Krylov methods).
*   **Fails? Use Homotopy:** If Newton diverges, try path-following from an easier problem.

## Exercises

### 1. Conceptual: Aitken's Acceleration
Fixed point iteration converges linearly. Aitken's $\Delta^2$ method accelerates this to quadratic. Research the formula and explain *why* it works (hint: it estimates the geometric decay of the error).

### 2. Applied: Implicit Yield Curve
Write a function `get_yield(price, coupon, face, T)` that handles an array of bonds. Use a loop with `brentq` to calculate the yield curve for a set of maturities: $T=[1, 2, 5, 10, 30]$. Plot the result.

### 3. Challenge: Multi-Market Equilibrium
Extend the CES code to 3 goods. You will now have two relative prices ($p_1, p_2$) to solve for. Use `scipy.optimize.root` to solve the system of 2 excess demand equations.